> ⚠️ **作業中 (Work in Progress)**: このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [エージェント 개요](#エージェント-개요)
- [ModelRouterAgent 作成](#modelrouteragent-作成)
- [FileSearchAgent 作成](#filesearchagent-作成)
- [WebSearchAgent 作成](#websearchagent-作成)
- [エージェント デプロイ 및 호출](#エージェント-デプロイ-및-호출)

## 🎯 学習目標

- Microsoft Foundry エージェント의 핵심 개념 이해
- Model Router 기반 エージェント 구축
- File Search 機能을 활용한 문서 기반 エージェント 作成
- Web Search 機能을 활용한 실時間 情報 検索 エージェント 作成
- エージェント デプロイ 및 프ログ래매틱 호출 방법 학습

## ⏱️ 予想所要時間

約30分

## 環境設定

먼저 プロジェクト エンドポイント를 設定합니다.

In [ ]:
# 環境 変数 ロード
import json
import os
import subprocess

# PATH 環境変数 設定 (Azure CLI를 찾을 수 있도록)
possible_paths = [
    "/opt/homebrew/bin",  # macOS (Apple Silicon)
    "/usr/local/bin",     # macOS (Intel) / Linux
    "/usr/bin",           # Linux / GitHub Codespaces
    "/home/linuxbrew/.linuxbrew/bin"  # Linux Homebrew
]

az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
except:
    pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
    paths_to_add.append(az_path)
else:
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get("PATH", ""):
            paths_to_add.append(path)

if paths_to_add:
    new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
    os.environ["PATH"] = new_path

# 前へ ノート북에서 保存한 設定 ファイル ロード
config_file = ".foundry_config.json"
try:
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    # 環境 変数 設定
    FOUNDRY_NAME = config.get("FOUNDRY_NAME")
    RESOURCE_GROUP = config.get("RESOURCE_GROUP")
    LOCATION = config.get("LOCATION")
    TENANT_ID = config.get("TENANT_ID")
    PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
    PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
    
    # 環境 変数로도 設定 (다른 도구들이 使用할 수 있도록)
    os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
    os.environ["LOCATION"] = LOCATION
    os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
    os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
    os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
    
    print(f"✅ 設定 ファイル '{config_file}'에서 環境 変数를 ロード했습니다.")
    print(f"\n📌 Foundry Name: {FOUNDRY_NAME}")
    print(f"📌 Resource Group: {RESOURCE_GROUP}")
    print(f"📌 Location: {LOCATION}")
    print(f"📌 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")
    
except FileNotFoundError:
    print(f"⚠️ '{config_file}' ファイル을 찾을 수 없습니다.")
    print("💡 01-setup.ipynb를 먼저 実行하여 環境을 設定하세요.")
    raise

# 必須パッケージのインストール
%pip install -q azure-ai-projects==2.0.0b2 azure-identity

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential

print(f"\n💡 使用할 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")

## ModelRouterAgent 作成

Model Router를 활용하여 지능적으로 モデル을 選択하는 エージェント를 만듭니다.

**Agent 構成:**
- **Model**: model-router (비용/품질/パフォーマンス 자동 최적화)
- **Instructions**: 질문 답변 エージェント
- **Tools**: 없음 (デフォルト 대화)

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()
client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# ModelRouterAgent 作成
ROUTER_INSTRUCTIONS = """당신은 질문에 답변하는 エージェント입니다.
リクエスト의 복잡도와 요구사항에 따라 가장 적절한 モデル을 使用하세요.
항상 명확하고 정확하며 도움이 되는 답변을 제공하세요."""

try:
    definition = PromptAgentDefinition(
        model="model-router",
        instructions=ROUTER_INSTRUCTIONS
    )
    
    agent_router = client.agents.create(
        name="ModelRouterAgent",
        definition=definition
    )
    
    print(f"✅ ModelRouterAgent 作成 完了!")
    print(f"   ID: {agent_router.id}")
    print(f"   Name: {agent_router.name}")
    print(f"   Model: {definition.model}")
    
except Exception as e:
    print(f"⚠️ エージェント 作成 失敗: {e}")
    print("\n💡 解決方法:")
    print("   1. Portal에서 'model-router' モデル이 デプロイ되었는지 確認")
    print("   2. モデル 名前이 정확한지 確認 (대소문자 구분)")

## FileSearchAgent 作成

ファイル 検索 機能을 활용하여 업ロード된 문서에서 情報를 찾는 エージェント입니다.

**Agent 構成:**
- **Model**: gpt-5.1
- **Tools**: file_search (ファイル 내용 検索)
- **Files**: knowledge-base.json 업ロード

**⚠️ 注意**: ファイル 업ロード는 Azure Portal(https://ai.azure.com)에서 더 편리합니다.
- Build > Agents > Create agent > Tools > File Search > Upload files

In [ ]:
# FileSearchAgent 作成

FILE_SEARCH_INSTRUCTIONS = """너는 Tools에 등록된 File search 기반으로 답변하는 エージェント입니다.

重要 규칙:
1. 반드시 업ロード된 ファイル의 내용을 기반으로만 답변하세요
2. ファイル에 없는 情報는 "제공된 문서에서 해당 情報를 찾을 수 없습니다"라고 답변하세요
3. 답변 시 출처 ファイル명을 언급하세요
4. 정확한 인용을 使用하세요"""

try:
    # FileSearchAgent 作成 (tools 없이 먼저 作成)
    definition = PromptAgentDefinition(
        model="gpt-5.1",
        instructions=FILE_SEARCH_INSTRUCTIONS
    )
    
    agent_filesearch = client.agents.create(
        name="FileSearchAgent",
        definition=definition
    )
    
    print(f"✅ FileSearchAgent 作成 完了!")
    print(f"   ID: {agent_filesearch.id}")
    print(f"   Name: {agent_filesearch.name}")
    print(f"   Model: {definition.model}")
    print(f"\n📋 次のステップ: Azure Portal에서 File Search 도구 追加")
    print(f"   1. Azure Portal (https://ai.azure.com) 접속")
    print(f"   2. Build > Agents > 'FileSearchAgent' 選択")
    print(f"   3. Tools 섹션에서 'File Search' 도구 追加")
    print(f"   4. knowledge-base.json ファイル 업ロード")
    print(f"   5. 업ロード 후 エージェント가 ファイル 내용을 検索할 수 있습니다")
    
except Exception as e:
    print(f"⚠️ エージェント 作成 失敗: {e}")
    print("\n💡 解決方法:")
    print("   1. 'gpt-5.1' モデル이 デプロイ되었는지 確認")
    print("   2. Portal에서 モデル 名前 確認 (대소문자 구분)")

## WebSearchAgent 作成

웹 検索 機能을 활용하여 실時間 情報를 제공하는 エージェント입니다.

**Agent 構成:**
- **Model**: gpt-4.1
- **Tools**: web_search (웹 検索)
- **機能**: 최신 뉴스, 날씨, 주식 情報 등

In [ ]:
# WebSearchAgent 作成

WEB_SEARCH_INSTRUCTIONS = """너는 Tools에 등록된 Web search 기반으로 답변하는 エージェント입니다.

重要 규칙:
1. 최신 情報가 필요한 질문에는 반드시 웹 検索을 使用하세요
2. 検索 結果를 기반으로 정확하고 최신의 情報를 제공하세요
3. 답변 시 출처 URL을 포함하세요
4. 여러 출처의 情報를 종합하여 균형잡힌 답변을 제공하세요
5. 検索 結果가 불충분하면 追加 検索을 수행하세요"""

try:
    definition = PromptAgentDefinition(
        model="gpt-4.1",
        instructions=WEB_SEARCH_INSTRUCTIONS,
        tools=[{"type": "web_search"}]
    )
    
    agent_websearch = client.agents.create(
        name="WebSearchAgent",
        definition=definition
    )
    
    print(f"✅ WebSearchAgent 作成 完了!")
    print(f"   ID: {agent_websearch.id}")
    print(f"   Name: {agent_websearch.name}")
    print(f"   Model: {definition.model}")
    print(f"   Tools: Web Search")
    
except Exception as e:
    print(f"⚠️ エージェント 作成 失敗: {e}")
    print("\n💡 解決方法:")
    print("   1. 'gpt-4.1' モデル이 デプロイ되었는지 確認")
    print("   2. Web Search가 プロジェクト에서 有効化되었는지 確認")

### 作成된 エージェント リスト 確認

## 作成된 エージェント 確認

In [ ]:
# 모든 エージェント リスト 取得
agents = client.agents.list()

print("=" * 80)
print("作成된 エージェント リスト")
print("=" * 80)

for agent in agents:
    print(f"\n📌 {agent.name}")
    print(f"   ID: {agent.id}")
    
    # versions에서 情報 추출
    if 'latest' in agent.versions:
        latest = agent.versions['latest']
        definition = latest.get('definition', {})
        
        model = definition.get('model', 'N/A')
        print(f"   Model: {model}")
        
        tools = definition.get('tools', [])
        if tools:
            tool_types = [t.get('type', 'unknown') if isinstance(t, dict) else str(t) for t in tools]
            print(f"   Tools: {', '.join(tool_types)}")
        else:
            print(f"   Tools: None")

print("\n✅ 포털 確認: https://ai.azure.com > Build > Agents")

## エージェント テスト (選択)

In [ ]:
# エージェント 객체 구조 確認 (디버깅용)
print("🔍 agent_router 객체 디버깅:")
print(f"Type: {type(agent_router)}")
print(f"\nAttributes:")
for attr in dir(agent_router):
    if not attr.startswith('_'):
        try:
            value = getattr(agent_router, attr)
            if not callable(value):
                print(f"  {attr}: {value}")
        except:
            pass

print(f"\n📋 Raw object:")
print(agent_router)

In [ ]:
# 간단한 エージェント テスト (選択사항)
# ModelRouterAgent와 대화

try:
    print("⏳ ModelRouterAgent 実行 중...")
    print(f"📍 Agent ID: {agent_router.id}")
    print(f"📍 Agent Name: {agent_router.name}")
    
    # SDK v2 - project client를 통해 OpenAI client 획득
    openai_client = client.get_openai_client()
    
    # Conversation 作成
    conversation = openai_client.conversations.create()
    print(f"💬 Conversation ID: {conversation.id}")
    
    # Responses API 호출
    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent": {"name": agent_router.name, "type": "agent_reference"}},
        input="Python에서 리스트를 ソート하는 방법을 알려주세요."
    )
    
    print("\n" + "=" * 80)
    print("🤖 ModelRouterAgent レスポンス:")
    print("=" * 80)
    print(response.output_text)
    
    # Conversation 정리
    openai_client.conversations.delete(conversation_id=conversation.id)
    print("\n✅ テスト 成功!")
    
except NameError:
    print("⚠️ agent_router 또는 client가 정의되지 않았습니다.")
    print("💡 위의 環境設定 및 ModelRouterAgent 作成 셀을 먼저 実行하세요.")
    
except Exception as e:
    print(f"⚠️ テスト 失敗: {e}")
    import traceback
    print(f"\n詳細 エラー:\n{traceback.format_exc()}")

In [ ]:
# WebSearchAgent テスト (選択사항)
# 최신 情報 検索

try:
    print("⏳ WebSearchAgent 実行 중 (웹 検索 중...)")
    print(f"📍 Agent ID: {agent_websearch.id}")
    print(f"📍 Agent Name: {agent_websearch.name}")
    
    # SDK v2 - project client를 통해 OpenAI client 획듍
    openai_client = client.get_openai_client()
    
    # Conversation 作成
    conversation = openai_client.conversations.create()
    print(f"💬 Conversation ID: {conversation.id}")
    
    # Responses API 호출
    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent": {"name": agent_websearch.name, "type": "agent_reference"}},
        input="오늘 서울 날씨는 어떤가요?"
    )
    
    print("\n" + "=" * 80)
    print("🌐 WebSearchAgent レスポンス:")
    print("=" * 80)
    print(response.output_text)
    
    # Conversation 정리
    openai_client.conversations.delete(conversation_id=conversation.id)
    print("\n✅ 웹 検索 도구가 실時間 情報를 가져왔습니다!")
    
except NameError:
    print("⚠️ agent_websearch 또는 client가 정의되지 않았습니다.")
    print("💡 위의 環境設定 및 WebSearchAgent 作成 셀을 먼저 実行하세요.")
    
except Exception as e:
    print(f"⚠️ テスト 失敗: {e}")
    import traceback
    print(f"\n詳細 エラー:\n{traceback.format_exc()}")

### ✅ 確認 사항

- エージェント가 成功적으로 게시되었는지 確認
- Python スクリプト가 エラー 없이 実行되는지 確認
- レスポンス이 예상대로 返却되는지 確認

## 📚 追加リソース

- [Microsoft Foundry Agents 개요](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/overview?view=foundry)
- [Agent SDK 문서](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/develop/sdk-overview?view=foundry&pivots=programming-language-python)
- [File Search 가이드](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/file-search?view=foundry&pivots=python)
- [Web Search 통합](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/web-search?view=foundry&pivots=python)

## 次のステップ

다양한 エージェント를 만들어보았습니다! 이제 Foundry IQ를 使用하여 고급 ナレッジベース을 구축해봅시다:

➡️ **[04. Foundry IQ](./04-foundry-iq.ipynb)**: AI Search와 Blob Storage를 활용한 ナレッジベース 구축을 학습합니다.